# SimpleNet Anomaly Detection: Master Reproduction Notebook

Reproduces the SimpleNet pipeline for the ADL 2025-2026 challenge.

Pipeline (single script `scripts/train_evaluate_simplenet_v2.py`):
1. **Training** — truncated-L1 discriminator on per-channel-scaled synthetic anomalies, plus an auxiliary margin loss on the few labeled anomaly masks.
2. **Local evaluation + PDF** — anomaly maps on the labeled training anomalies, for manual inspection (NOTE: training-set overlap, optimistic).
3. **Submission** — per-class percentile-clipped + multi-view sample gate → q8rle CSV/ZIP.

**Note:** Designed to run on Google Colab or locally with the correct environment.

## 1. Setup and Environment

In [1]:
!nvidia-smi

Sun May 17 17:05:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.142                Driver Version: 580.142        CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4070 ...    Off |   00000000:01:00.0 Off |                  N/A |
|  0%   37C    P8             11W /  220W |     100MiB /  12282MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# If running on Colab, uncomment and run this:
# !pip install timm opencv-python pandas tqdm matplotlib scipy torchvision

In [2]:
import os
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")
print(f"Current working directory: {os.getcwd()}")

Current working directory: /home/max/Documents/ghi8/adl/anomaly-detection


## 2. Execution Pipeline

The v2 script runs training, local evaluation + PDF generation, and submission encoding in one pass.

Tweakable flags:
- `--epochs` (default 15)
- `--lr` (default 1e-3)
- `--backbone` (default `wide_resnet50_2`)
- `--noise_std` (default 0.015) — synthetic-anomaly noise scale (multiplied by per-channel std)
- `--th` (default 0.5) — truncated-L1 hinge threshold
- `--w_aux` (default 0.05) — weight on the auxiliary supervised mask loss; set to 0 to disable
- `--view_gate_alpha` (default 0.5) — sample-level multi-view multiplicative gate strength; set to 0 to disable
- `--classes class_01 class_02 ...` (default: all classes found under `PATH`)

Outputs:
- `artifacts/simplenet_v2/metrics.csv`
- `artifacts/simplenet_v2/predictions.pdf`
- `submission/submission_simplenet_v2.csv`
- `submission/submission_simplenet_v2.zip`

In [3]:
# Train + local eval + PDF + submission (one pass)
%run scripts/train_evaluate_simplenet_v2.py

Processing 8 classes using SimpleNet (wide_resnet50_2)

>>> Class: class_01
Training on 2600 good + 25 labeled anomaly samples...


KeyboardInterrupt: 

## 3. Kaggle Submission

Uncomment and run the following cell to submit the results directly to Kaggle.

In [ ]:
# !kaggle competitions submit -c adl-2025-2026-anomaly-detection -f submission/submission_simplenet_v2.zip -m "SimpleNet v2 (truncated-L1 + aux mask + view gate)"